In [2]:
# Install dependencies if needed:
# !pip install -r requirements.txt

## Scratchpad 

1. Let them know v3.1 deprecated - only have v5 and it requires bearer token 
2. Look for alternatives? 
3. Design decision — Selective source ingestion

The REST Countries ingestion process requests only attributes required for country identification and currency resolution. Geographic and demographic attributes are intentionally excluded because they do not support the defined exchange-rate use case. The selected API response is preserved in its original structure before downstream transformations.



In [3]:
import requests
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Find the repository .env from the notebook's working directory.
env_path = next(
    (folder / ".env" for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / ".env").is_file()),
    None,
)
if env_path is not None:
    load_dotenv(env_path)

API_KEY = os.getenv("countries_api_key")

In [4]:
headers = {"Authorization": f"Bearer {API_KEY}",
            "Accept": "application/json",}

params = {
    "response_fields": (
        "names.common,"
        "names.official,"
        "codes.alpha_2,"
        "codes.alpha_3,"
        "region,"
        "subregion,"
        "capitals,"
        "population,"
        "currencies"
    ),
    "limit": 100,
    "offset": 0
}

In [5]:

URL = "https://api.restcountries.com/countries/v5"

FIELDS = [
    "names.common",
    "names.official",
    "codes.alpha_2",
    "codes.alpha_3",
    "region",
    "subregion",
    "capitals",
    "population",
    "currencies",
]

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Accept": "application/json",
}

countries = []
offset = 0
limit = 100

with requests.Session() as session:
    while True:
        params = {
            "response_fields": ",".join(FIELDS),
            "limit": limit,
            "offset": offset,
        }

        response = session.get(
            URL,
            headers=headers,
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        payload = response.json()
        data = payload["data"]

        records = data["objects"]
        meta = data["meta"]

        countries.extend(records)

        print(
            f"Fetched {len(records)} records "
            f"(total so far: {len(countries)})"
        )

        if not meta["more"]:
            break

        if not records:
            raise ValueError(
                "API indicates more data, but returned an empty page"
            )

        offset += len(records)

# Verify completeness
if len(countries) != meta["total"]:
    raise ValueError(
        f"Expected {meta['total']} countries, "
        f"received {len(countries)}"
    )

# Save the source records without flattening
with open("countries_raw.json", "w", encoding="utf-8") as f:
    json.dump(countries, f, indent=2, ensure_ascii=False)

print(f"\nSuccessfully fetched {len(countries)} countries")
print("Saved to countries_raw.json")

# Inspect Peru
peru = next(
    (
        country
        for country in countries
        if country.get("codes", {}).get("alpha_3") == "PER"
    ),
    None,
)

print("\nPeru record:")
print(json.dumps(peru, indent=2, ensure_ascii=False))


Fetched 100 records (total so far: 100)
Fetched 100 records (total so far: 200)
Fetched 54 records (total so far: 254)

Successfully fetched 254 countries
Saved to countries_raw.json

Peru record:
{
  "names": {
    "common": "Peru",
    "official": "Republic of Peru"
  },
  "codes": {
    "alpha_2": "PE",
    "alpha_3": "PER"
  },
  "capitals": [
    {
      "attributes": {
        "administrative": false,
        "constitutional": false,
        "executive": false,
        "judicial": false,
        "legislative": false,
        "primary": true
      },
      "coordinates": {
        "lat": -12.05,
        "lng": -77.05
      },
      "name": "Lima"
    }
  ],
  "region": "Americas",
  "subregion": "South America",
  "currencies": [
    {
      "code": "PEN",
      "name": "Peruvian sol",
      "symbol": "S/"
    }
  ],
  "population": 34157732,
  "_meta": {
    "lastUpdatedTimestamp": 1791319714
  }
}


In [ ]:
# MEta has:

# total
# count
# limit
# offset
# more
# request_id
# duration

## Alternative? 

In [6]:
import requests


countries_url = "https://countries.dev/countries"
countries_fields = ["name", "alpha2Code", "alpha3Code", "currencies"]
with requests.Session() as session:
    response = session.get(
            countries_url,
            params={"fields": ",".join(countries_fields)},
            timeout=30,
        )

In [8]:
# response.json()

## Frankfurter

In [9]:
from datetime import date, timedelta


In [10]:

API_URL = "https://api.frankfurter.dev/v2/rates"
START_DATE = date(2026, 1, 1)
params = {"base": "USD"}

In [15]:
response = requests.get(
        API_URL,
        params={"base": "USD", "date": "2026-11-03"},
        timeout=60,
    )

In [17]:
response_json = response.json()

In [18]:
len(response_json)

170

In [19]:
response_json

[{'date': '2026-11-03', 'base': 'USD', 'quote': 'AED', 'rate': 3.6725},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'AFN', 'rate': 65.066},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'ALL', 'rate': 81.18},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'AMD', 'rate': 360.24},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'ANG', 'rate': 1.79},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'AOA', 'rate': 919.34},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'ARS', 'rate': 1543.02},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'AUD', 'rate': 1.4291},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'AWG', 'rate': 1.7895},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'AZN', 'rate': 1.6992},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'BAM', 'rate': 1.7469},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'BBD', 'rate': 2.059},
 {'date': '2026-11-02', 'base': 'USD', 'quote': 'BCH', 'rate': 0.003633},
 {'date': '2026-11-03', 'base': 'USD', 'quote': 'BDT', 'rate': 12

# ??

In [21]:
!pip install duckdb

Defaulting to user installation because normal site-packages is not writeable
  Using cached duckdb-2.0.0-cp312-abi3-win_amd64.whl.metadata (4.1 kB)
Using cached duckdb-2.0.0-cp312-abi3-win_amd64.whl (15.4 MB)



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from pathlib import Path
import duckdb

warehouse_path = Path("data/warehouse/branch.duckdb").resolve()

# Read-only access prevents accidental changes.
con = duckdb.connect(str(warehouse_path), read_only=True)

In [23]:
display(con.sql("""
    SELECT table_schema, table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'core'
    ORDER BY table_name, ordinal_position
"""))

table_schemaVARCHAR,table_nameVARCHAR,column_nameVARCHAR,data_typeVARCHAR
core,countries,country_code,VARCHAR
core,countries,country_name,VARCHAR
core,country_currencies,country_code,VARCHAR
core,country_currencies,currency_code,VARCHAR
core,exchange_rates,rate_date,DATE
core,exchange_rates,base_currency,VARCHAR
core,exchange_rates,quote_currency,VARCHAR
core,exchange_rates,rate,DOUBLE


In [27]:
display(con.sql("""select * from core.country_currencies
where country_code in (SELECT country_code FROM core.country_currencies 
group by 1 having count(*) > 1 ) """))


country_codeVARCHAR,currency_codeVARCHAR
BLR,BYN
BLR,BYR
BRN,BND
BRN,SGD
BTN,BTN
BTN,INR
COK,CKD
COK,NZD
CUB,CUC
CUB,CUP


In [ ]:
# con.close()